In [ ]:
# pyright: reportUnusedImport=false, reportMissingImports=false, reportMissingModuleSource=false
import tensorflow as tf
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input, Lambda
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input


BASE_DIR = Path(__file__).parent
TRAIN_DIR = BASE_DIR / "dataset" / "train"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10


train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    seed=100,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="validation",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    seed=100,
)

class_names = train_dataset.class_names
print("Classes:", class_names)

for images, labels in train_dataset.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)


data_augmentation = Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
])

train_dataset = (
    train_dataset
    .cache()
    .shuffle(100)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .cache()
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)


resnet_base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
)

resnet_base.trainable = False

model = Sequential([
    Input(shape=(224, 224, 3)),
    data_augmentation,
    Lambda(preprocess_input),
    resnet_base,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])

model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=["accuracy"],
)

history = model.fit(
    train_dataset,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        EarlyStopping(
            monitor="val_accuracy",
            patience=5,
            restore_best_weights=True,
        )
    ],
)

model.save(BASE_DIR / "hotdog_resnet50_classifier.keras")
print("Model saved as hotdog_resnet50_classifier.keras")